# Stage 1c/1d calibration diagnostics (2026-07-17)

Diagnostic/visualization companion to the Stage 1b re-re-calibration triggered by the 2026-07-16
feed-mechanism fixes (`bcGateAssim` made permanent, `rankByVelocity` caching, chrono/For-You
`rootPostId` dedup). Full narrative + decisions: `docs/report/2026-07-17-stage1b-redo2-plan.md`
and `docs/report/2026-07-15-calibration-decisions.md`.

This notebook does **not** run any simulation — it only loads the CSV logs already produced by:
- `scripts/stage1c_bo_calibration.py` -> `docs/report/stage1c_bo_evaluations.csv`
- `scripts/stage1d_bo_calibration.py` -> `docs/report/stage1d_bo_evaluations.csv` (widened box, warm-started from stage1c)
- `scripts/stage1c_confirm_seed80013.py` -> `docs/report/stage1c_confirm_seed80013.csv` (boundary candidate, 5 seeds)
- `scripts/stage1c_confirm_seed80023.py` -> `docs/report/stage1c_confirm_seed80023.csv` (interior candidate, 5 seeds)

The BO loop itself stays in plain scripts (long-running, nohup'd) for reproducibility; this notebook
is the exploratory-diagnostics layer on top, per the transparency-recording policy in the plan doc.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

REPORT_DIR = Path("../docs/report") if Path("../docs/report").exists() else Path("docs/report")

GINI_TARGET = 0.8
GAMMA_TARGET = 2.276
ASSORT_FLOOR = 0.6

stage1c = pd.read_csv(REPORT_DIR / "stage1c_bo_evaluations.csv")
stage1d = pd.read_csv(REPORT_DIR / "stage1d_bo_evaluations.csv")
stage1c["stage"] = "1c"
stage1d["stage"] = "1d"
all_evals = pd.concat([stage1c, stage1d], ignore_index=True)
all_evals.sort_values("loss").head(10)

## Convergence: best-loss-so-far by evaluation order

Stage 1c ran without a warm start (dynamics changed since the prior stage1b-redo run, so its
evaluated points weren't reusable — see plan doc). Stage 1d warm-starts from all 25 stage1c points
(same dynamics, only the box widened), so its "best so far" line starts at stage1c's final best.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for stage, df, offset in [("1c", stage1c, 0), ("1d", stage1d, len(stage1c))]:
    d = df.sort_values(["round", "seed"]).reset_index(drop=True)
    running_best = d["loss"].cummin()
    ax.plot(range(offset, offset + len(d)), running_best, marker="o", label=f"stage{stage}")

ax.set_xlabel("evaluation index (stage1c then stage1d)")
ax.set_ylabel("best loss so far")
ax.set_yscale("log")
ax.axvline(len(stage1c) - 0.5, color="gray", linestyle="--", linewidth=1, label="stage1c -> 1d handoff")
ax.legend()
ax.set_title("BO convergence: stage1c (no warm start) -> stage1d (warm-started, widened box)")
plt.tight_layout()
plt.show()

## Parameter vs. loss, with search-box boundaries marked

The stage1c best point (seed 80013) sat on 3 of 4 box boundaries (`max_follow` upper, `follow_prob`
lower, `hop` upper) — flagged in red. Stage1d widened exactly those three bounds; none of its new
points beat stage1c's best, which argues the boundary-pinning wasn't a truncation artifact. The
5-seed confirmation (loaded below) is what actually disqualified the boundary point, on
`opinionAssortativity` grounds rather than loss.

In [ ]:
BOUNDS_1C = {"max_follow": (15, 100), "follow_prob": (0.01, 0.3), "alpha": (0.0, 1.0), "hop": (1, 5)}
BOUNDS_1D = {"max_follow": (10, 200), "follow_prob": (0.003, 0.3), "alpha": (0.0, 1.0), "hop": (1, 8)}

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
params = ["max_follow", "follow_prob", "alpha", "hop"]

for ax, p in zip(axes, params):
    ax.scatter(stage1c[p], stage1c["loss"], label="stage1c", alpha=0.7)
    ax.scatter(stage1d[p], stage1d["loss"], label="stage1d", alpha=0.7, marker="^")
    lo, hi = BOUNDS_1C[p]
    ax.axvline(lo, color="red", linestyle=":", linewidth=1)
    ax.axvline(hi, color="red", linestyle=":", linewidth=1)
    if p == "follow_prob":
        ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(p)
    ax.set_ylabel("loss")
    ax.set_title(p)

axes[0].legend()
plt.suptitle("Red dotted lines = stage1c's original (narrower) search-box bounds")
plt.tight_layout()
plt.show()

## 5-seed confirmation: boundary candidate vs. interior candidate

This is the key result. Single-seed BO loss estimates turned out to be extremely noisy — both
candidates' 5-seed mean loss is 10-25x their single-seed BO value. The boundary point
(`follow_prob=0.01`) fails the `opinionAssortativity` floor in 4 of 5 seeds (one seed collapses to
0.163); the interior point (`follow_prob=0.0145`) clears it in 4 of 5. **Decision (2026-07-17,
`calibration-decisions.md`): the interior point is the calibrated Stage-1b value**, chosen for
floor-stability over the boundary point's lower nominal loss.

In [ ]:
confirm_boundary = pd.read_csv(REPORT_DIR / "stage1c_confirm_seed80013.csv")
confirm_interior = pd.read_csv(REPORT_DIR / "stage1c_confirm_seed80023.csv")

summary = pd.DataFrame({
    "candidate": ["boundary (seed80013)", "interior (seed80023, CALIBRATED)"],
    "max_follow": [100.0, 29.32],
    "follow_prob": [0.01, 0.0145],
    "alpha": [0.884, 0.763],
    "hop": [5, 4],
    "loss_mean": [confirm_boundary["loss"].mean(), confirm_interior["loss"].mean()],
    "loss_std": [confirm_boundary["loss"].std(), confirm_interior["loss"].std()],
    "assort_mean": [confirm_boundary["opinion_assortativity"].mean(), confirm_interior["opinion_assortativity"].mean()],
    "assort_std": [confirm_boundary["opinion_assortativity"].std(), confirm_interior["opinion_assortativity"].std()],
    "floor_cleared": [
        (confirm_boundary["opinion_assortativity"] >= ASSORT_FLOOR).sum(),
        (confirm_interior["opinion_assortativity"] >= ASSORT_FLOOR).sum(),
    ],
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
labels = ["boundary\n(seed80013)", "interior\n(seed80023, CALIBRATED)"]
for i, (df, label) in enumerate(zip([confirm_boundary, confirm_interior], labels)):
    ys = df["opinion_assortativity"].values
    xs = np.full_like(ys, i, dtype=float) + np.random.uniform(-0.05, 0.05, size=len(ys))
    ax.scatter(xs, ys, s=60, alpha=0.8)
    ax.scatter([i], [ys.mean()], s=200, marker="_", color="black")

ax.axhline(ASSORT_FLOOR, color="red", linestyle="--", label=f"floor = {ASSORT_FLOOR}")
ax.set_xticks([0, 1])
ax.set_xticklabels(labels)
ax.set_ylabel("opinionAssortativity (per seed)")
ax.set_title("5-seed spread: boundary candidate fails the floor 4/5 times")
ax.legend()
plt.tight_layout()
plt.show()

## Conclusion

Calibrated Stage-1b values (2026-07-17, see `calibration-decisions.md`):
`max_follow=29.3, follow_prob=0.0145, alpha=0.763 (anchor, not fixed), hop=4 (anchor, not fixed)`.

Open follow-up: Stage 2 (`bcDecRate`, stubbornness, `comfortPpFactor`, `feedCapacity`,
`initialPostProb`) needs restarting on top of these values, and — per the noise finding above —
should average 2-3 seeds per BO-evaluated point rather than 1.